# TimeSeries analysis and forecasting with the current `mnplib` API

This notebook demonstrates the current `TimeSeries` class.

The estimator treats forecasting as **minimum-nescience model selection over lagged representations**:

```python
ordered series -> lagged supervised representation
candidate forecaster -> selected subset, predictions, model_string
metrics -> deficiency, surplus, inaccuracy, surfeit
select candidate with minimum nescience
```

The time-series estimator follows the current explicit-artifact API. Each candidate is evaluated through:

- `subset`: selected lagged features;
- `predictions`: one-step predictions on the lagged representation;
- `model_string`: a canonical description of the forecasting rule.

This notebook covers:

- target auto-lag analysis;
- exogenous cross-lag analysis;
- candidate comparison;
- selected model explanation;
- canonical model strings;
- recursive forecasting;
- model-family restrictions;
- exogenous-variable forecasting;
- rolling nescience for regime-change detection.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mnplib.timeseries import TimeSeries

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (11, 4)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 2. Helper functions

In [ ]:
def plot_series(y, title, *, ylabel="Value"):
    """Plot a one-dimensional series."""
    pd.Series(y).plot(figsize=(11, 4))
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_candidate_nescience(results, title):
    """Plot scalar nescience for fitted time-series candidates."""
    results.set_index("model_name")["nescience"].plot(kind="bar", figsize=(12, 4))
    plt.title(title)
    plt.ylabel("Nescience")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_candidate_components(results, title):
    """Plot the four nescience components for fitted candidates."""
    component_cols = ["deficiency", "surplus", "inaccuracy", "surfeit"]
    results.set_index("model_name")[component_cols].plot(kind="bar", figsize=(12, 5))
    plt.title(title)
    plt.ylabel("Component value")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_forecast(y, forecast, title):
    """Plot observed history and recursive forecasts."""
    history = pd.Series(y)
    future_index = np.arange(len(y), len(y) + len(forecast))
    plt.figure(figsize=(11, 4))
    plt.plot(history.index, history.values, label="Observed")
    plt.plot(future_index, forecast, marker="o", label="Forecast")
    plt.axvline(len(y) - 1, linestyle="--")
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.legend()
    plt.tight_layout()
    plt.show()


def print_components(model):
    """Print selected model components."""
    for name, value in model.components().items():
        print(f"{name:>10}: {value:.6f}")


def compact_results(results):
    """Return the most useful result-table columns."""
    columns = [
        "model_name",
        "model_family",
        "nescience",
        "estimator_score",
        "is_reliable",
        "failure_reason",
        "deficiency",
        "surplus",
        "miscoding",
        "inaccuracy",
        "surfeit",
        "n_selected_features",
        "description_length",
        "selected_feature_names",
    ]
    return results[columns]

## 3. Example 1: a synthetic autoregressive series

We start with a simple series where the current value depends mainly on recent past target values. This is the canonical case for an autoregressive candidate.


In [ ]:
n = 180
noise = rng.normal(0.0, 0.35, size=n)
y = np.zeros(n)

for t in range(2, n):
    y[t] = 0.65 * y[t - 1] - 0.25 * y[t - 2] + noise[t]

plot_series(y, "Synthetic autoregressive series")

## 4. Fit the current `TimeSeries` estimator

The estimator builds the lagged representation internally, evaluates the configured candidate families, and selects the candidate with minimum nescience.


In [ ]:
ts = TimeSeries(
    window_size=8,
    models=("autoregressive", "moving_average", "exponential_smoothing"),
    moving_average_windows=(1, 2, 3, 5, 8),
    smoothing_alphas=(0.2, 0.5, 0.8),
    n_bins="auto",
    min_improvement=0.0,
    description_precision=4,
    random_state=RANDOM_SEED,
    verbose=1,
)

ts.fit(y)

## 5. Selected model and main diagnostics

In [ ]:
print("Selected model:", ts.model_name_)
print("Model family:", ts.best_result_.model_family)
print("Window size:", ts.window_size_)
print("Selected feature names:", ts.selected_feature_names_)
print("Native estimator score:", ts.score(y))
print("Nescience:", ts.nescience_score())

print("\nComponents:")
print_components(ts)

## 6. Candidate comparison table

Reliable candidates are shown first and sorted by ascending nescience, so the selected candidate appears first.


In [ ]:
results = ts.results_dataframe()
compact_results(results)

In [ ]:
plot_candidate_nescience(results, "Candidate nescience for the autoregressive example")
plot_candidate_components(results, "Candidate components for the autoregressive example")

## 7. Auto-miscoding: target lag diagnostics

`auto_lag_analysis()` evaluates how informative each lag of the target is with respect to the current target value. Lower miscoding is better.


In [ ]:
auto_lags = ts.auto_lag_analysis(max_lag=12)
auto_lags

In [ ]:
auto_lags.set_index("feature_name")[["deficiency", "surplus", "miscoding"]].plot(
    kind="bar",
    figsize=(12, 4),
)
plt.title("Target lag diagnostics")
plt.ylabel("Value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Canonical model string

The selected forecasting rule is represented by a canonical string. This is the string consumed by the `Surfeit` metric.


In [ ]:
model_string = ts.model_string()

print(model_string[:2500])
print("\nDescription length in bytes:", len(model_string.encode("utf-8")))

## 9. Structured explanation

`explain()` gives a compact interpretation of the selected model and its dominant source of nescience.


In [ ]:
explanation = ts.explain()

print("Profile:", explanation["profile"])
print("Dominant component:", explanation["dominant_component"])
print("Recommendation:", explanation["recommendation"])

print("\nSelected lags:")
for lag in explanation["selected_lags"]:
    print(lag)

## 10. Recursive forecasting

`forecast(steps)` produces recursive future predictions. Each forecasted value is appended to the target history and used to build the next lagged row.


In [ ]:
forecast = ts.forecast(steps=20)
forecast

In [ ]:
plot_forecast(y, forecast, "Recursive forecast for the autoregressive example")

## 11. Restricting model families

The estimator lets you compare only selected forecasting families. This is useful when you want a controlled demonstration of a specific type of forecaster.


In [ ]:
family_rows = []

for family in [
    ("autoregressive",),
    ("moving_average",),
    ("exponential_smoothing",),
]:
    model = TimeSeries(
        window_size=8,
        models=family,
        moving_average_windows=(1, 2, 3, 5, 8),
        smoothing_alphas=(0.2, 0.5, 0.8),
        n_bins="auto",
        description_precision=4,
        random_state=RANDOM_SEED,
    )
    model.fit(y)
    family_rows.append(
        {
            "families": ", ".join(family),
            "selected_model": model.model_name_,
            "nescience": model.nescience_score(),
            "score": model.score(y),
            **model.components(),
        }
    )

family_comparison = pd.DataFrame(family_rows)
family_comparison

In [ ]:
family_comparison.set_index("families")[["deficiency", "surplus", "inaccuracy", "surfeit"]].plot(
    kind="bar",
    figsize=(10, 4),
)
plt.title("Selected model components by restricted family")
plt.ylabel("Component value")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 12. Example 2: forecasting with exogenous variables

Now we create a target series affected by an external temperature-like signal. The `TimeSeries` class builds lagged features for both the target and the exogenous variables.


In [ ]:
n = 220
time = np.arange(n)

temperature = np.sin(2 * np.pi * time / 24) + 0.15 * rng.normal(size=n)
promotion = ((time % 40) < 6).astype(float)

y_exog = np.zeros(n)
noise = 0.18 * rng.normal(size=n)

for t in range(3, n):
    y_exog[t] = (
        0.50 * y_exog[t - 1]
        + 0.35 * temperature[t - 2]
        + 0.20 * promotion[t - 1]
        + noise[t]
    )

X_exog = pd.DataFrame(
    {
        "temperature": temperature,
        "promotion": promotion,
    }
)

plot_series(y_exog, "Synthetic series with exogenous drivers")
X_exog.head()

## 13. Fit with exogenous variables

In [ ]:
ts_exog = TimeSeries(
    window_size=10,
    models=("autoregressive", "moving_average", "exponential_smoothing"),
    moving_average_windows=(1, 2, 3, 5, 10),
    smoothing_alphas=(0.2, 0.5, 0.8),
    n_bins="auto",
    min_improvement=0.0,
    description_precision=4,
    random_state=RANDOM_SEED,
)

ts_exog.fit(y_exog, X=X_exog)

print("Selected model:", ts_exog.model_name_)
print("Model family:", ts_exog.best_result_.model_family)
print("Selected features:", ts_exog.selected_feature_names_)
print("Score:", ts_exog.score(y_exog, X=X_exog))
print("Nescience:", ts_exog.nescience_score())

print("\nComponents:")
print_components(ts_exog)

In [ ]:
exog_results = ts_exog.results_dataframe()
compact_results(exog_results)

In [ ]:
plot_candidate_nescience(exog_results, "Candidate nescience with exogenous variables")
plot_candidate_components(exog_results, "Candidate components with exogenous variables")

## 14. Cross-miscoding: exogenous lag diagnostics

`cross_lag_analysis(attribute)` shows how lagged values of an exogenous attribute relate to the target.


In [ ]:
temperature_lags = ts_exog.cross_lag_analysis("temperature", max_lag=12)
promotion_lags = ts_exog.cross_lag_analysis("promotion", max_lag=12)

temperature_lags

In [ ]:
combined_lags = pd.concat(
    [
        temperature_lags.assign(source="temperature"),
        promotion_lags.assign(source="promotion"),
    ],
    ignore_index=True,
)

combined_lags.pivot(index="lag", columns="source", values="miscoding").plot(
    marker="o",
    figsize=(10, 4),
)
plt.title("Cross-lag miscoding for exogenous variables")
plt.ylabel("Miscoding")
plt.xlabel("Lag")
plt.tight_layout()
plt.show()

## 15. Forecasting with future exogenous values

When exogenous variables are used, future values can be supplied through `X_future`. If they are omitted, the class repeats the last observed exogenous row.


In [ ]:
future_steps = 24
future_time = np.arange(n, n + future_steps)
X_future = pd.DataFrame(
    {
        "temperature": np.sin(2 * np.pi * future_time / 24),
        "promotion": ((future_time % 40) < 6).astype(float),
    }
)

forecast_exog = ts_exog.forecast(steps=future_steps, X_future=X_future)
plot_forecast(y_exog, forecast_exog, "Recursive forecast with future exogenous values")

## 16. Inspect the exogenous model string

In [ ]:
print(ts_exog.model_string()[:3000])

## 17. Example 3: rolling nescience for regime-change detection

Rolling nescience can be used as a lightweight way to inspect structural changes. Fit a compact `TimeSeries` model on rolling windows and record the selected candidate's nescience.


In [ ]:
n = 260
regime = np.zeros(n)
regime_noise = 0.25 * rng.normal(size=n)

for t in range(2, n):
    if t < 130:
        regime[t] = 0.75 * regime[t - 1] + regime_noise[t]
    else:
        regime[t] = -0.35 * regime[t - 1] + 0.55 * regime[t - 2] + regime_noise[t]

plot_series(regime, "Synthetic series with a regime change")

In [ ]:
window = 80
step = 10
rolling_rows = []

for start in range(0, len(regime) - window + 1, step):
    end = start + window
    segment = regime[start:end]

    model = TimeSeries(
        window_size=8,
        models=("autoregressive", "moving_average", "exponential_smoothing"),
        moving_average_windows=(1, 2, 3, 5, 8),
        smoothing_alphas=(0.2, 0.5, 0.8),
        n_bins=4,
        description_precision=4,
        random_state=RANDOM_SEED,
    )
    model.fit(segment)

    rolling_rows.append(
        {
            "start": start,
            "end": end,
            "center": start + window // 2,
            "selected_model": model.model_name_,
            "nescience": model.nescience_score(),
            "inaccuracy": model.components()["inaccuracy"],
            "surfeit": model.components()["surfeit"],
            "deficiency": model.components()["deficiency"],
            "surplus": model.components()["surplus"],
        }
    )

rolling = pd.DataFrame(rolling_rows)
rolling

In [ ]:
rolling.set_index("center")[["nescience", "inaccuracy", "surfeit"]].plot(
    marker="o",
    figsize=(11, 4),
)
plt.axvline(130, linestyle="--", label="True regime change")
plt.title("Rolling nescience around a regime change")
plt.xlabel("Window center")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

## 18. Summary

The current `TimeSeries` class exposes a compact and consistent API:

```python
ts = TimeSeries(...)
ts.fit(y, X=None)

ts.results_dataframe()
ts.nescience_score()
ts.components()
ts.explain()
ts.model_string()
ts.forecast(steps, X_future=None)
ts.auto_lag_analysis()
ts.cross_lag_analysis(attribute)
ts.lag_analysis()
```

Forecasting candidates are evaluated through explicit artifacts:

```python
subset, predictions, model_string
```

This keeps the metric layer independent from the forecasting implementation and makes the time-series class easier to maintain.
